In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')
print("All imports ready ✅")

All imports ready ✅


In [ ]:
# Load the CarDekho dataset
df = pd.read_csv('cardekho_dataset.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (15411, 14)
Columns: ['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [ ]:
df = df.drop(columns=['Unnamed: 0', 'car_name', 'model'])
print(f"Shape after dropping: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")


Shape after dropping: (15411, 11)
Remaining columns: ['brand', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


In [ ]:
df

,brand,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...
15406,Hyundai,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Maruti,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Skoda,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,Mahindra,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [ ]:
df['fuel_type'].value_counts()

,count
fuel_type,
Petrol,7643
Diesel,7419
CNG,301
LPG,44
Electric,4


In [ ]:
# One-hot encode all categorical columns
df = pd.get_dummies(df, columns=['brand', 'seller_type', 'fuel_type',
                                  'transmission_type'], drop_first=True)

print(f"Shape after encoding: {df.shape}")
print(f"Sample of new columns: {[c for c in df.columns if '_' in c][:10]}")


Shape after encoding: (15411, 45)
Sample of new columns: ['vehicle_age', 'km_driven', 'max_power', 'selling_price', 'brand_BMW', 'brand_Bentley', 'brand_Datsun', 'brand_Ferrari', 'brand_Force', 'brand_Ford']


In [ ]:
df

,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price,brand_BMW,brand_Bentley,brand_Datsun,...,brand_Toyota,brand_Volkswagen,brand_Volvo,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,transmission_type_Manual
0,9,120000,19.70,796,46.30,5,120000,False,False,False,...,False,False,False,True,False,False,False,False,True,True
1,5,20000,18.90,1197,82.00,5,550000,False,False,False,...,False,False,False,True,False,False,False,False,True,True
2,11,60000,17.00,1197,80.00,5,215000,False,False,False,...,False,False,False,True,False,False,False,False,True,True
3,9,37000,20.92,998,67.10,5,226000,False,False,False,...,False,False,False,True,False,False,False,False,True,True
4,6,30000,22.77,1498,98.59,5,570000,False,False,False,...,False,False,False,False,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,9,10723,19.81,1086,68.05,5,250000,False,False,False,...,False,False,False,False,False,False,False,False,True,True
15407,2,18000,17.50,1373,91.10,7,925000,False,False,False,...,False,False,False,False,False,False,False,False,True,True
15408,6,67000,21.14,1498,103.52,5,425000,False,False,False,...,False,False,False,False,False,True,False,False,False,True
15409,5,3800000,16.00,2179,140.00,7,1225000,False,False,False,...,False,False,False,False,False,True,False,False,False,True


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 45 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   vehicle_age                   15411 non-null  int64  
 1   km_driven                     15411 non-null  int64  
 2   mileage                       15411 non-null  float64
 3   engine                        15411 non-null  int64  
 4   max_power                     15411 non-null  float64
 5   seats                         15411 non-null  int64  
 6   selling_price                 15411 non-null  int64  
 7   brand_BMW                     15411 non-null  bool   
 8   brand_Bentley                 15411 non-null  bool   
 9   brand_Datsun                  15411 non-null  bool   
 10  brand_Ferrari                 15411 non-null  bool   
 11  brand_Force                   15411 non-null  bool   
 12  brand_Ford                    15411 non-null  bool   
 13  b

In [ ]:
X = df.drop(columns=['selling_price'])
y = df['selling_price']

print(f"Features: {X.shape[0]} rows × {X.shape[1]} columns")
print(f"Target: selling_price — ₹{y.min():,} to ₹{y.max():,}")
print(f"Target mean: ₹{y.mean():,.0f}, median: ₹{y.median():,.0f}")


Features: 15411 rows × 44 columns
Target: selling_price — ₹40,000 to ₹39,500,000
Target mean: ₹774,971, median: ₹556,000


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test:  {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")


Train: 12328 rows (80%)
Test:  3083 rows (20%)


In [ ]:
numeric_cols = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

In [ ]:
X_train

,vehicle_age,km_driven,mileage,engine,max_power,seats,brand_BMW,brand_Bentley,brand_Datsun,brand_Ferrari,...,brand_Toyota,brand_Volkswagen,brand_Volvo,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,transmission_type_Manual
11210,0.323969,0.349100,-2.050819,1.756765,2.681685,-0.403824,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
1347,-1.337798,-1.069394,0.985661,-0.547081,-0.382744,-0.403824,False,False,False,False,...,False,False,False,True,False,False,False,False,True,True
10363,-1.337798,-1.163564,-0.177042,0.893542,3.296910,-0.403824,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
316,0.323969,0.178369,-0.465315,0.024564,0.396229,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
10638,1.321030,0.585469,0.149668,-0.550917,-0.502047,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5191,0.323969,1.702310,0.248161,-0.453086,-0.270460,2.070500,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
13418,1.653383,0.084198,-0.876105,0.218310,0.066393,-0.403824,False,False,False,False,...,False,True,False,False,False,False,False,False,True,True
5390,0.323969,-0.833967,0.185702,-0.932654,-0.779483,-0.403824,False,False,False,False,...,False,False,False,True,False,False,False,False,True,True
860,-1.337798,-0.951680,-0.273133,-0.550917,-0.432805,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Model trained ✅")
print(f"Number of coefficients: {len(model.coef_)}")
print(f"Intercept: ₹{model.intercept_:,.0f}")

Model trained ✅
Number of coefficients: 44
Intercept: ₹1,324,659


In [ ]:
model.coef_

array([-2.27679116e+05, -4.61441160e+04, -1.61215737e+04,  7.81262973e+04,
        3.35373847e+05,  3.61447730e+04,  4.05858369e+05,  3.66129519e+06,
       -7.09797494e+05,  3.35712157e+07, -6.84161881e+05, -6.05929376e+05,
       -6.34461348e+05, -5.81777427e+05, -5.39248828e+05, -6.72399952e+05,
        3.89829810e+05, -3.30328774e+05, -1.88217266e+05,  1.37973826e+06,
        2.43851701e+06, -3.34953252e+05, -7.57351045e+05, -5.09629233e+05,
        3.05324950e+06, -3.72529030e-09,  2.12192715e+05,  5.81734336e+05,
       -6.92637218e+05,  1.83113118e+06, -6.41815125e+05,  1.83478941e+07,
       -6.76533456e+05, -8.25722441e+05, -4.17293997e+05, -6.20822305e+05,
        1.07324564e+06, -1.20906369e+04, -8.11214717e+04,  4.90755661e+04,
        2.93031384e+05,  2.77714921e+05, -4.11245704e+04, -5.92492214e+04])

In [ ]:
# Predictions on both sets
y_train_pred = model.predict(X_train)
print(y_train_pred)

[3412193.03734085  538816.57194792 3107972.2792696  ...  315413.96838231
  792625.57941124 1096655.71792075]


In [ ]:
print(y_train)

11210    1825000
1347      515000
10363    7500000
316       435000
10638     200000
          ...   
5191      665000
13418     249000
5390      250000
860       620000
7270      960000
Name: selling_price, Length: 12328, dtype: int64


In [ ]:
# Metrics on TRAIN
train_mae = mean_absolute_error(y_train, y_train_pred)
print(train_mae)

199219.72900407415


In [ ]:

y_test_pred = model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
print(test_mae)

212761.44410581293


In [ ]:
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
y_test_pred = model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
print(test_mae)

669950.2365915277


In [ ]:
df['selling_price'].mean()

np.float64(774971.1164103562)

In [ ]:
df['base_pred'] = 774971.1164103562

In [ ]:
base_mae = mean_absolute_error(df['selling_price'], df['base_pred'])
print(base_mae)

452669.921078294


In [ ]:
train_r2 = r2_score(y_train, y_train_pred)
print(train_r2)

0.8015753782439604


In [ ]:
# Build a coefficient table
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})
coef_df

,Feature,Coefficient
0,vehicle_age,-2.276791e+05
1,km_driven,-4.614412e+04
2,mileage,-1.612157e+04
3,engine,7.812630e+04
4,max_power,3.353738e+05
5,seats,3.614477e+04
6,brand_BMW,4.058584e+05
7,brand_Bentley,3.661295e+06
8,brand_Datsun,-7.097975e+05
9,brand_Ferrari,3.357122e+07


In [ ]:
# Build a coefficient table
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})
coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coef', ascending=False)

print("Top 15 Most Impactful Features (by coefficient magnitude):")
print("=" * 65)
for _, row in coef_df.head(15).iterrows():
    direction = "↑" if row['Coefficient'] > 0 else "↓"
    bar_len = int(row['Abs_Coef'] / coef_df['Abs_Coef'].max() * 30)
    bar = "█" * bar_len
    print(f"  {direction} {row['Feature']:30s} {bar} ({row['Coefficient']:>+12,.0f})")

print(f"\n  Intercept: ₹{model.intercept_:,.0f}")


Top 15 Most Impactful Features (by coefficient magnitude):
  ↑ brand_Ferrari                  ██████████████████████████████ ( +33,571,216)
  ↑ brand_Rolls-Royce              ████████████████ ( +18,347,894)
  ↑ brand_Bentley                  ███ (  +3,661,295)
  ↑ brand_Maserati                 ██ (  +3,053,250)
  ↑ brand_Lexus                    ██ (  +2,438,517)
  ↑ brand_Porsche                  █ (  +1,831,131)
  ↑ brand_Land Rover               █ (  +1,379,738)
  ↑ brand_Volvo                     (  +1,073,246)
  ↓ brand_Tata                      (    -825,722)
  ↓ brand_Mahindra                  (    -757,351)
  ↓ brand_Datsun                    (    -709,797)
  ↓ brand_Nissan                    (    -692,637)
  ↓ brand_Force                     (    -684,162)
  ↓ brand_Skoda                     (    -676,533)
  ↓ brand_Isuzu                     (    -672,400)

  Intercept: ₹1,324,659


In [ ]:
# Pick one car from the test set
sample_idx = X_test.index[0]

In [ ]:
sample_idx

np.int64(3334)

In [ ]:
# Pick one car from the test set
sample_idx = X_test.index[0]
sample_features = X_test.loc[sample_idx]
sample_actual = y_test.loc[sample_idx]

In [ ]:
sample_features

,3334
vehicle_age,1.985737
km_driven,0.413795
mileage,0.149668
engine,-0.550917
max_power,-0.502047
seats,-0.403824
brand_BMW,False
brand_Bentley,False
brand_Datsun,False
brand_Ferrari,False


In [ ]:
sample_actual

np.int64(190000)

In [ ]:
type(X_test)

pandas.core.frame.DataFrame

In [ ]:
X_test

,vehicle_age,km_driven,mileage,engine,max_power,seats,brand_BMW,brand_Bentley,brand_Datsun,brand_Ferrari,...,brand_Toyota,brand_Volkswagen,brand_Volvo,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,transmission_type_Manual
3334,1.985737,0.413795,0.149668,-0.550917,-0.502047,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
10928,-0.673091,0.060655,1.838470,-0.453086,-0.616670,-0.403824,False,False,False,False,...,False,False,False,True,False,True,False,False,False,True
2518,0.323969,0.955277,0.248161,-0.453086,-0.271396,2.070500,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
11322,-1.670152,-1.198878,-0.321179,0.026483,0.444184,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
9394,1.653383,0.154826,-0.008882,-1.320145,-1.264645,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1943,0.988676,-0.410198,0.560457,-0.547081,-0.619010,-0.403824,False,False,False,False,...,False,True,False,False,False,True,False,False,False,True
10471,-1.337798,-1.116479,-0.393247,-0.547081,-0.272799,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
13098,-1.337798,-1.253027,-0.393247,-0.547081,-0.272799,-0.403824,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
6545,-1.005445,-0.598540,0.185702,-0.932654,-0.779483,-0.403824,False,False,False,False,...,False,False,False,True,False,False,False,False,True,True


In [ ]:
# Method 1: model.predict()
# Pick one car from the test set
sample_idx = X_test.index[0]
sample_features = X_test.loc[sample_idx]
sample_actual = y_test.loc[sample_idx]

# Method 1: model.predict()
sklearn_prediction = model.predict(X_test.loc[[sample_idx]])[0]

print(sklearn_prediction)

-57120.863150607096


In [ ]:
# Manul calcualtion

In [ ]:
sample_features

,3334
vehicle_age,1.985737
km_driven,0.413795
mileage,0.149668
engine,-0.550917
max_power,-0.502047
seats,-0.403824
brand_BMW,False
brand_Bentley,False
brand_Datsun,False
brand_Ferrari,False


In [ ]:
Manual_prediction = 0


contributions = []
for feat, value in sample_features.items():
    coef = model.coef_[list(X_train.columns).index(feat)]
    contribution = coef * value
    contributions.append((feat, value, coef, contribution))

for i in contributions:
    print(i)

('vehicle_age', np.float64(1.9857368532491522), np.float64(-227679.11587518523), np.float64(-452110.8111085394))
('km_driven', np.float64(0.4137954453676014), np.float64(-46144.1160234312), np.float64(-19094.225041009984))
('mileage', np.float64(0.14966790175177855), np.float64(-16121.57370910439), np.float64(-2412.8821099782917))
('engine', np.float64(-0.5509174225989446), np.float64(78126.29726953516), np.float64(-43041.13832893127))
('max_power', np.float64(-0.5020465790486293), np.float64(335373.84695451916), np.float64(-168373.2925658949))
('seats', np.float64(-0.40382371071387335), np.float64(36144.77304341482), np.float64(-14596.116373302553))
('brand_BMW', np.False_, np.float64(405858.3691318681), np.float64(0.0))
('brand_Bentley', np.False_, np.float64(3661295.186986896), np.float64(0.0))
('brand_Datsun', np.False_, np.float64(-709797.4940396062), np.float64(-0.0))
('brand_Ferrari', np.False_, np.float64(33571215.66884778), np.float64(0.0))
('brand_Force', np.False_, np.float6

In [ ]:
len(contributions)

44

In [ ]:
Manual_prediction = 0
for i in contributions:
    Manual_prediction = Manual_prediction + i[3]

Manual_prediction = Manual_prediction + model.intercept_
print(Manual_prediction)

-57120.863150607096


In [ ]:
# Pick one car from the test set
sample_idx = X_test.index[0]
sample_features = X_test.loc[sample_idx]
sample_actual = y_test.loc[sample_idx]

# Method 1: model.predict()
sklearn_prediction = model.predict(X_test.loc[[sample_idx]])[0]

# Method 2: Manual calculation
manual_prediction = 0.0
print("Manual Prediction Breakdown:")
print("=" * 70)

# Show the top contributing features (non-zero ones)
contributions = []
for feat, val in sample_features.items():
    coef = model.coef_[list(X_train.columns).index(feat)]
    contribution = coef * val
    contributions.append((feat, val, coef, contribution))

# Sort by absolute contribution
contributions.sort(key=lambda x: abs(x[3]), reverse=True)

# Show top 10 contributors
for feat, val, coef, contrib in contributions[:10]:
    manual_prediction += contrib
    print(f"  {feat:30s}  value={val:>8.3f} × coef={coef:>12,.0f} = ₹{contrib:>12,.0f}")

# Add remaining features
remaining = sum(c[3] for c in contributions[10:])
manual_prediction += remaining
print(f"  {'(remaining features)':30s}  {'':>8s}   {'':>12s}   ₹{remaining:>12,.0f}")

# Add intercept
manual_prediction += model.intercept_
print(f"  {'+ Intercept':30s}  {'':>8s}   {'':>12s}   ₹{model.intercept_:>12,.0f}")
print("=" * 70)
print(f"  Manual prediction:   ₹{manual_prediction:>12,.0f}")
print(f"  sklearn prediction:  ₹{sklearn_prediction:>12,.0f}")
print(f"  Actual price:        ₹{sample_actual:>12,.0f}")
print(f"  Error:               ₹{abs(sample_actual - sklearn_prediction):>12,.0f}")
print(f"\n  Match? {abs(manual_prediction - sklearn_prediction) < 1}  ✅")


Manual Prediction Breakdown:
  brand_Hyundai                   value=   1.000 × coef=    -581,777 = ₹    -581,777
  vehicle_age                     value=   1.986 × coef=    -227,679 = ₹    -452,111
  max_power                       value=  -0.502 × coef=     335,374 = ₹    -168,373
  transmission_type_Manual        value=   1.000 × coef=     -59,249 = ₹     -59,249
  engine                          value=  -0.551 × coef=      78,126 = ₹     -43,041
  fuel_type_Petrol                value=   1.000 × coef=     -41,125 = ₹     -41,125
  km_driven                       value=   0.414 × coef=     -46,144 = ₹     -19,094
  seats                           value=  -0.404 × coef=      36,145 = ₹     -14,596
  mileage                         value=   0.150 × coef=     -16,122 = ₹      -2,413
  brand_BMW                       value=   0.000 × coef=     405,858 = ₹           0
  (remaining features)                                      ₹           0
  + Intercept                                  